# Waste Classifier with YOLO

Classify images as organic, inorganic, or empty with YOLO26.

# 1. Install the libraries

In [ ]:
import sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    %pip install -q ultralytics==8.4.175
else:
    %pip install -q ultralytics==8.4.175 opencv-python==5.0.0.93 matplotlib==3.11.2 scikit-learn==1.9.1 pillow==12.3.0

# 2. Imports

In [ ]:
import csv
import random
import shutil
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import torch
from PIL import Image
from sklearn.metrics import ConfusionMatrixDisplay, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from ultralytics import YOLO

print("PyTorch", torch.__version__, "| GPU available:", torch.cuda.is_available())

# 3. Configuration

Main settings for data, training, and predictions.

In [ ]:
CONFIGURATION = {
    "DATA_DIR": "data",                              # your photos, one subfolder per class
    "YOLO_DATA_DIR": "datasets/sampah_yolo",         # the train/val/test copy that YOLO trains on
    "CLASS_NAMES": ["organik", "anorganik", "kosong"],
    "BACKGROUND_CLASS": "kosong",                    # "nothing here" class -> no output
    "MIN_IMAGES_PER_CLASS": 10,                      # classes with fewer photos are skipped
    "MODEL": "yolo26n-cls.pt",                       # yolo26s-cls.pt = more accurate but slower
    "IM_SIZE": 224,
    "BATCH_SIZE": 32,                                # lower it (e.g. 16) if you run out of memory
    "N_EPOCHS": 30,
    "PATIENCE": 10,                                  # stop early after this many epochs without improvement
    "SPLIT": (0.7, 0.2, 0.1),                        # train / validation / test
    "SEED": 42,
    "CONF_THRESHOLD": 0.6,                           # min confidence to say "there's trash"
    "RUNS_DIR": "runs",                              # YOLO saves charts and logs here
    "RUN_NAME": "sampah",
    "MODEL_PATH": "models/yolo_sampah.pt",
}

random.seed(CONFIGURATION["SEED"])

# On Colab, read photos from and save results to Google Drive so nothing gets lost.
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_FOLDER = "/content/drive/MyDrive/klasifikasi-sampah-notebook"   # change if you put it elsewhere
    CONFIGURATION["DATA_DIR"] = f"{DRIVE_FOLDER}/data"
    CONFIGURATION["RUNS_DIR"] = f"{DRIVE_FOLDER}/runs"
    CONFIGURATION["MODEL_PATH"] = f"{DRIVE_FOLDER}/models/yolo_sampah.pt"
    # the YOLO copy stays on Colab's local disk, which is much faster than Drive
    CONFIGURATION["YOLO_DATA_DIR"] = "/content/datasets/sampah_yolo"

# 4. Check your photos

Only valid images from classes with enough samples are used.

In [ ]:
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
data_dir = Path(CONFIGURATION["DATA_DIR"])


def list_photos(folder):
    return sorted(p for p in folder.rglob("*") if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS)


def is_valid_image(path):
    try:
        with Image.open(path) as image:
            image.verify()
        return True
    except Exception:
        return False


photos_per_class = {}
for name in CONFIGURATION["CLASS_NAMES"]:
    folder = data_dir / name
    folder.mkdir(parents=True, exist_ok=True)  # creates the empty folders on the first run
    photos = list_photos(folder)
    good = [p for p in photos if is_valid_image(p)]
    photos_per_class[name] = good
    note = f"  ({len(photos) - len(good)} broken file(s) skipped)" if len(good) < len(photos) else ""
    print(f"{name:<10} {len(good):>5} photos{note}")

CLASS_NAMES = [n for n, p in photos_per_class.items() if len(p) >= CONFIGURATION["MIN_IMAGES_PER_CLASS"]]
skipped = [n for n in CONFIGURATION["CLASS_NAMES"] if n not in CLASS_NAMES]
if skipped:
    print(f"\nSkipping {skipped}: fewer than {CONFIGURATION['MIN_IMAGES_PER_CLASS']} photos.")

assert len(CLASS_NAMES) >= 2, f"Need at least 2 classes with photos. Add photos to {data_dir.resolve()} and run this cell again."
NUM_CLASSES = len(CLASS_NAMES)
print("\nClasses used:", CLASS_NAMES)

# 5. Split the data

Images are split into training, validation, and test sets. Folder names define the classes.

In [ ]:
paths, labels = [], []
for index, name in enumerate(CLASS_NAMES):
    paths += photos_per_class[name]
    labels += [index] * len(photos_per_class[name])

train_frac, val_frac, test_frac = CONFIGURATION["SPLIT"]
seed = CONFIGURATION["SEED"]

# First cut: train vs the rest. Second cut: the rest -> val and test.
# stratify=... keeps the class balance the same in every split.
train_paths, rest_paths, train_labels, rest_labels = train_test_split(
    paths, labels, train_size=train_frac, stratify=labels, random_state=seed)
val_paths, test_paths, val_labels, test_labels = train_test_split(
    rest_paths, rest_labels, train_size=val_frac / (val_frac + test_frac), stratify=rest_labels, random_state=seed)

print(f"{'class':<10}{'train':>8}{'val':>8}{'test':>8}")
for index, name in enumerate(CLASS_NAMES):
    counts = [split.count(index) for split in (train_labels, val_labels, test_labels)]
    print(f"{name:<10}" + "".join(f"{c:>8}" for c in counts))

yolo_dir = Path(CONFIGURATION["YOLO_DATA_DIR"])
if yolo_dir.exists():
    shutil.rmtree(yolo_dir)

for split, split_paths, split_labels in [("train", train_paths, train_labels),
                                         ("val", val_paths, val_labels),
                                         ("test", test_paths, test_labels)]:
    for i, (path, label) in enumerate(zip(split_paths, split_labels)):
        target = yolo_dir / split / CLASS_NAMES[label] / f"{i:05d}{path.suffix.lower()}"
        target.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(path, target)

print("\nYOLO dataset ready at", yolo_dir.resolve())

# 6. Preview the training data

A sample preview helps check that images match their labels.

In [ ]:
sample = random.sample(list(zip(train_paths, train_labels)), k=min(12, len(train_paths)))
plt.figure(figsize=(12, 8))
for i, (path, label) in enumerate(sample):
    plt.subplot(3, 4, i + 1)
    plt.imshow(Image.open(path).convert("RGB"))
    plt.title(CLASS_NAMES[label])
    plt.axis("off")
plt.suptitle("Training photos")
plt.tight_layout()
plt.show()

# 7. Load a pretrained YOLO classifier

Pretrained weights give the classifier a useful starting point.

In [ ]:
Path(CONFIGURATION["MODEL_PATH"]).parent.mkdir(parents=True, exist_ok=True)
model = YOLO(str(Path(CONFIGURATION["MODEL_PATH"]).parent / CONFIGURATION["MODEL"]))
print("Task:", model.task)

# 8. Train

The model learns from the training images; validation guides selection of the best weights.

In [ ]:
model.train(
    data=str(yolo_dir.resolve()),
    epochs=CONFIGURATION["N_EPOCHS"],
    imgsz=CONFIGURATION["IM_SIZE"],
    batch=CONFIGURATION["BATCH_SIZE"],
    patience=CONFIGURATION["PATIENCE"],
    seed=CONFIGURATION["SEED"],
    project=str(Path(CONFIGURATION["RUNS_DIR"]).resolve()),
    name=CONFIGURATION["RUN_NAME"],
    exist_ok=True,   # overwrite the previous run with the same name
    plots=True,
)

RUN_DIR = Path(model.trainer.save_dir)
shutil.copy2(model.trainer.best, CONFIGURATION["MODEL_PATH"])
print("Charts and logs:", RUN_DIR)
print("Best model saved to:", CONFIGURATION["MODEL_PATH"])

# 9. Training curves

Loss and accuracy curves show learning progress and generalization.

In [ ]:
with open(RUN_DIR / "results.csv") as f:
    rows = [{key.strip(): value for key, value in row.items()} for row in csv.DictReader(f)]

epochs = [int(r["epoch"]) for r in rows]
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(epochs, [float(r["train/loss"]) for r in rows], label="train")
plt.plot(epochs, [float(r["val/loss"]) for r in rows], label="validation")
plt.title("Loss")
plt.xlabel("epoch")
plt.xticks(epochs)
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(epochs, [float(r["metrics/accuracy_top1"]) for r in rows])
plt.title("Validation accuracy")
plt.xlabel("epoch")
plt.xticks(epochs)

plt.tight_layout()
plt.show()

# 10. Evaluate on the test set

Test metrics measure performance on unseen images; the confusion matrix shows class mix-ups.

In [ ]:
model = YOLO(CONFIGURATION["MODEL_PATH"])
names = [model.names[i] for i in sorted(model.names)]

metrics = model.val(
    data=str(yolo_dir.resolve()), split="test", imgsz=CONFIGURATION["IM_SIZE"],
    batch=CONFIGURATION["BATCH_SIZE"], project=str(Path(CONFIGURATION["RUNS_DIR"]).resolve()),
    name=CONFIGURATION["RUN_NAME"] + "_test", exist_ok=True, verbose=False,
)
print(f"Top-1 accuracy: {metrics.top1:.3f}\n")

test_dir = yolo_dir / "test"
test_images = [p for name in names for p in sorted((test_dir / name).glob("*")) if (test_dir / name).exists()]
y_true = [p.parent.name for p in test_images]
y_pred = [r.names[r.probs.top1]
          for r in model.predict([str(p) for p in test_images], imgsz=CONFIGURATION["IM_SIZE"],
                                 stream=True, verbose=False)]

print(classification_report(y_true, y_pred, labels=names, digits=3, zero_division=0))
ConfusionMatrixDisplay(confusion_matrix(y_true, y_pred, labels=names), display_labels=names).plot(cmap="Blues")
plt.title("Confusion matrix (test set)")
plt.show()

# 11. View sample predictions

Compare predictions with true labels on test images.

In [ ]:
sample = random.sample(test_images, k=min(12, len(test_images)))
plt.figure(figsize=(12, 9))
for i, path in enumerate(sample):
    result = model.predict(str(path), imgsz=CONFIGURATION["IM_SIZE"], verbose=False)[0]
    pred, conf, real = result.names[result.probs.top1], float(result.probs.top1conf), path.parent.name
    plt.subplot(3, 4, i + 1)
    plt.imshow(Image.open(path).convert("RGB"))
    plt.title(f"real: {real}\npred: {pred} ({conf:.2f})", color="green" if pred == real else "red")
    plt.axis("off")
plt.tight_layout()
plt.show()

# 12. Set the final output

Predictions for the empty class or below the confidence threshold produce no output.

In [ ]:
def decide(probs, names):
    """probs -> (output, top_label, top_conf). output is None when there's no trash."""
    index = int(np.argmax(probs))
    label, conf = names[index], float(probs[index])
    if label == CONFIGURATION["BACKGROUND_CLASS"] or conf < CONFIGURATION["CONF_THRESHOLD"]:
        return None, label, conf
    return label, label, conf


def predict_frame(model, frame_bgr):
    """Classify one OpenCV frame (BGR) and apply the output rule. YOLO takes BGR frames directly."""
    result = model.predict(frame_bgr, imgsz=CONFIGURATION["IM_SIZE"], verbose=False)[0]
    probs = result.probs.data.cpu().numpy()
    return decide(probs, result.names)

# 13. Try one photo

View the predicted class and confidence for a single image.

In [ ]:
TEST_IMAGE = str(test_images[0])   # e.g. "my_photo.jpg"

frame = cv2.imread(TEST_IMAGE)
output, top_label, top_conf = predict_frame(model, frame)
if output:
    print(output)

plt.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
plt.title(f"top guess: {top_label} ({top_conf:.2f})")
plt.axis("off")
plt.show()

# 14. Reload the saved model

Load saved weights for inference without retraining.

In [ ]:
model = YOLO(CONFIGURATION["MODEL_PATH"])
print("Loaded model with classes:", list(model.names.values()))

# 15. Live webcam

Classify webcam frames in real time.

In [ ]:
RUN_WEBCAM = False   # set to True on your laptop

if RUN_WEBCAM and not IN_COLAB:
    cap = cv2.VideoCapture(0)   # 0 = default webcam, try 1 if you have another one
    last_output = None
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        output, top_label, top_conf = predict_frame(model, frame)

        if output != last_output:   # only print when the result changes
            if output:
                print(output)
            last_output = output

        text = f"{output} {top_conf:.2f}" if output else f"no trash ({top_label} {top_conf:.2f})"
        cv2.putText(frame, text, (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 255, 0), 2)
        cv2.imshow("Waste classifier - press Q to quit", frame)
        if cv2.waitKey(1) & 0xFF == ord("q"):
            break
    cap.release()
    cv2.destroyAllWindows()
else:
    print("Webcam is off. Set RUN_WEBCAM = True (laptop only) to start it.")